Limpeza de dados
---

*600 promoções realizadas nos últimos anos*

> Mulheres demoram mais para ser promovidas? Compare o meses_no_cargo_anterior entre homens e mulheres no histórico de promoções.

> No histórico de promoções, compare meses_no_cargo_anterior por sexo — mulheres esperam ~40% mais?

> Funcionários com satisfação ≤ 2 e sem promoção são os mais propensos a sairpas









`Importação das bibliotecas`

In [ ]:
# Bibliotecas principais
import pandas as pd
import numpy as np

# Visualização
import seaborn as sns
import matplotlib.pyplot as plt

# Configuração visual
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

`Carregar os datasets`

In [10]:
url = "https://github.com/marciacastro03/projeto_07-Dossi-Exodus-Crise-de-Talentos-e-Discrimina-o/blob/main/data/projeto_07_historico_promocoes.csv"

df = pd.read_csv(url)

HTTPError: HTTP Error 429: Too Many Requests

`Limpeza de dados`

In [11]:
# ==========================================
# PROJETO PEOPLE ANALYTICS
# Investigação de Promoções e Possível Gap de Gênero
# ==========================================

# ------------------------------------------
# 1. IMPORTAÇÃO DE BIBLIOTECAS
# ------------------------------------------

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind

sns.set(style="whitegrid")

plt.rcParams["figure.figsize"] = (10,6)


# ------------------------------------------
# 2. CARREGAR DATASET
# ------------------------------------------

df = pd.read_csv("/content/projeto_07_historico_promocoes.csv")

print("Dimensão do dataset:")
print(df.shape)

print("\nColunas:")
print(df.columns)

print("\nPrimeiras linhas:")
print(df.head())


# ------------------------------------------
# 3. INSPEÇÃO INICIAL
# ------------------------------------------

print("\nInformações do dataset:")
print(df.info())

print("\nResumo estatístico:")
print(df.describe())


# ------------------------------------------
# 4. LIMPEZA DE DADOS
# ------------------------------------------

# remover duplicatas
df = df.drop_duplicates()

# verificar valores nulos
print("\nValores nulos:")
print(df.isnull().sum())

# remover nulos relevantes
df = df.dropna(subset=["sexo","meses_no_cargo_anterior"])

# converter tipos
df["data_promocao"] = pd.to_datetime(df["data_promocao"], errors="coerce")

df["sexo"] = df["sexo"].astype("category")
df["departamento"] = df["departamento"].astype("category")

# criar variável de ano
df["ano_promocao"] = df["data_promocao"].dt.year


# ------------------------------------------
# 5. DETECÇÃO DE OUTLIERS
# ------------------------------------------

plt.figure()

sns.boxplot(x=df["meses_no_cargo_anterior"])

plt.title("Distribuição de meses antes da promoção")

plt.show()

# remover valores extremos
df = df[df["meses_no_cargo_anterior"] < 120]


# ------------------------------------------
# 6. ANÁLISE PRINCIPAL - TEMPO ATÉ PROMOÇÃO
# ------------------------------------------

tempo_promocao = df.groupby("sexo")["meses_no_cargo_anterior"].agg(
    media="mean",
    mediana="median",
    desvio="std",
    quantidade="count"
)

print("\nTempo médio até promoção por gênero:")
print(tempo_promocao)


# ------------------------------------------
# 7. TESTE ESTATÍSTICO (T-TEST)
# ------------------------------------------

homens = df[df["sexo"]=="M"]["meses_no_cargo_anterior"]
mulheres = df[df["sexo"]=="F"]["meses_no_cargo_anterior"]

t_stat, p_value = ttest_ind(homens, mulheres, equal_var=False)

print("\nResultado do teste t:")

print("T-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Diferença estatisticamente significativa.")
else:
    print("Diferença NÃO estatisticamente significativa.")


# ------------------------------------------
# 8. BOXPLOT COMPARATIVO
# ------------------------------------------

plt.figure()

sns.boxplot(
    data=df,
    x="sexo",
    y="meses_no_cargo_anterior"
)

plt.title("Tempo até promoção por gênero")

plt.xlabel("Sexo")
plt.ylabel("Meses no cargo")

plt.show()


# ------------------------------------------
# 9. VIOLINPLOT
# ------------------------------------------

plt.figure()

sns.violinplot(
    data=df,
    x="sexo",
    y="meses_no_cargo_anterior"
)

plt.title("Distribuição do tempo até promoção")

plt.show()


# ------------------------------------------
# 10. COUNT PLOT DE PROMOÇÕES
# ------------------------------------------

plt.figure()

sns.countplot(
    data=df,
    x="sexo"
)

plt.title("Número de promoções por gênero")

plt.show()


# ------------------------------------------
# 11. ANÁLISE POR DEPARTAMENTO
# ------------------------------------------

tempo_departamento = df.groupby(
    ["departamento","sexo"]
)["meses_no_cargo_anterior"].mean().reset_index()

print("\nTempo médio por departamento:")
print(tempo_departamento)


sns.catplot(
    data=tempo_departamento,
    x="departamento",
    y="meses_no_cargo_anterior",
    hue="sexo",
    kind="bar",
    height=6,
    aspect=2
)

plt.xticks(rotation=45)

plt.title("Tempo médio até promoção por departamento")

plt.show()


# ------------------------------------------
# 12. DIFERENÇA SALARIAL
# ------------------------------------------

salario_genero = df.groupby("sexo")[
    ["salario_anterior","salario_novo","aumento_percentual"]
].mean()

print("\nMédia salarial por gênero:")
print(salario_genero)


plt.figure()

sns.boxplot(
    data=df,
    x="sexo",
    y="aumento_percentual"
)

plt.title("Distribuição de aumento percentual")

plt.show()


# ------------------------------------------
# 13. AVALIAÇÃO DE PERFORMANCE
# ------------------------------------------

plt.figure()

sns.boxplot(
    data=df,
    x="sexo",
    y="avaliacao_no_momento"
)

plt.title("Avaliação de performance no momento da promoção")

plt.show()


# ------------------------------------------
# 14. HEATMAP DE CORRELAÇÃO
# ------------------------------------------

corr = df[
[
"meses_no_cargo_anterior",
"salario_anterior",
"salario_novo",
"aumento_percentual",
"avaliacao_no_momento"
]
].corr()

plt.figure()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlação entre variáveis de promoção")

plt.show()


# ------------------------------------------
# 15. PROPORÇÃO DE PROMOÇÕES POR DEPARTAMENTO
# ------------------------------------------

promo_dept = pd.crosstab(
    df["departamento"],
    df["sexo"],
    normalize="index"
)

promo_dept.plot(
    kind="bar",
    stacked=True
)

plt.title("Proporção de promoções por gênero e departamento")

plt.ylabel("Proporção")

plt.show()


# ------------------------------------------
# 16. ANÁLISE DE COORTE (ANO)
# ------------------------------------------

coorte = df.groupby(
    ["ano_promocao","sexo"]
)["meses_no_cargo_anterior"].mean().reset_index()

plt.figure()

sns.lineplot(
    data=coorte,
    x="ano_promocao",
    y="meses_no_cargo_anterior",
    hue="sexo",
    marker="o"
)

plt.title("Tempo médio até promoção ao longo dos anos")

plt.show()


# ------------------------------------------
# 17. FUNÇÃO PARA CUSTO DE TURNOVER
# ------------------------------------------

def custo_turnover(numero_saidas, custo_por_saida=45000):
    return numero_saidas * custo_por_saida


print("\nExemplo de custo de turnover:")

print("20 saídas =", custo_turnover(20))

SyntaxError: invalid syntax (2531833103.py, line 1)